# Is one epoch enough? Training-budget ablation for the LoRA extractor

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rkesseku/clinical-lora-extractor/blob/main/notebooks/ablation_colab.ipynb)

The reported run trained for **1 epoch on 2,000 BC5CDR sentences**. This notebook tests that choice:

* a grid of **epochs (1, 2, 3) x training size (2,000 vs. the full train split)**, each trained from a fresh base model;
* every configuration is scored on a fixed sample of the **validation** split, and the best one is chosen there;
* the **test** set is used once, for the selected configuration and the reference recipe (2,000 sentences, 1 epoch), with a paired bootstrap of the difference.

Runtime: **Runtime > Change runtime type > T4 GPU**. Cell 4 prints a time estimate for the grid before anything trains;
shrink `EPOCHS` / `N_TRAIN` there if it is too long for one session.

In [ ]:
# 1. GPU check
import subprocess

import torch

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "no nvidia-smi")
assert torch.cuda.is_available(), "No GPU found. Use Runtime > Change runtime type > T4 GPU."
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

In [ ]:
# 2. Install the package from GitHub (Colab already ships a CUDA build of torch; it is kept as is).
# The second line force-reinstalls the repo itself, so a reused runtime always gets the latest code.
import sys

if "clinical_lora" in sys.modules:
    raise RuntimeError(
        "An older clinical_lora is already imported in this session. "
        "Use Runtime > Restart session (or Disconnect and delete runtime), then Run all again."
    )
# Colab preinstalls an old torchao (0.10); recent peft refuses to import alongside torchao < 0.16.
# This project does not use torchao, so remove it.
%pip uninstall -q -y torchao
%pip install -q "git+https://github.com/rkesseku/clinical-lora-extractor.git@main"
%pip install -q --force-reinstall --no-deps "git+https://github.com/rkesseku/clinical-lora-extractor.git@main"

import clinical_lora

print("clinical_lora", clinical_lora.__version__)

In [ ]:
# 3. Grid and sizes (edit here)
from clinical_lora.ablation import AblationConfig

EPOCHS = (1.0, 2.0, 3.0)
N_TRAIN = (2000, None)  # None = the full training split

acfg = AblationConfig(
    base_model="Qwen/Qwen2.5-1.5B-Instruct",
    epochs=EPOCHS,
    n_train=N_TRAIN,
    reference=(2000, 1.0),  # the recipe reported in the README
    n_val=300,  # validation sentences used for selection
    n_test=200,  # same 200 test sentences as the main notebook (seed 0)
    seed=0,
    out_dir="results_ablation",
    adapter_root="adapters/ablation",
)
acfg

In [ ]:
# 4. Load BC5CDR once and estimate the run time
from clinical_lora.data import load_bc5cdr

splits = load_bc5cdr()
print({k: len(v) for k, v in splits.items()})

# The reported run trained 2,000 sentences for 1 epoch in 4.8 min on a T4; evaluation of 300
# validation sentences adds roughly 2-3 min per configuration. Treat this as a rough guide.
n_all = len(splits["train"])
units = sum((n or n_all) / 2000 * e for n in N_TRAIN for e in EPOCHS)
n_cfg = len(N_TRAIN) * len(EPOCHS)
print(f"{n_cfg} configurations, about {units * 4.8 + n_cfg * 2.5 + 10:.0f} minutes in total")

In [ ]:
# 5. Run the grid, select on validation, evaluate the winner and the reference once on test
from clinical_lora.ablation import run_ablation

results = run_ablation(acfg, splits=splits)
print("selected:", results["selected"], "| reference:", results["reference"])

In [ ]:
# 6. Report and plot
from IPython.display import Image, Markdown, display

display(Markdown(open("results_ablation/ablation.md").read()))
display(Image("results_ablation/ablation_val_f1.png"))

In [ ]:
# 7. Zip the results for download (send results_ablation/ablation.json back to update the README)
import shutil

shutil.make_archive("results_ablation", "zip", "results_ablation")
try:
    from google.colab import files

    files.download("results_ablation.zip")
except Exception:
    print("results_ablation.zip written to the working directory")